In [1]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline
import torch


/home/ubuntu/myenv/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load tained model

In [3]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPretraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [4]:
model.load_state_dict(torch.load('state.pt'))

<All keys matched successfully>

In [5]:
model = model.eval()

### load and Tokenize MRPC test

In [6]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

dataset_test = load_dataset('glue', 'mrpc', split='test')

#Encode dataset
def encode(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length')

dataset_test = dataset_test.map(encode, batched=True)
#Rename label to labels
dataset_test = dataset_test.map(lambda rows: {'labels': rows['label']}, batched=True)

#Set dataset format and dataloader
dataset_test.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=16)

Found cached dataset glue (/home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


In [7]:
results = []
for i, batch in enumerate(test_loader):
    _, outputs = model(**batch)
    outputs = torch.max(outputs, 1).indices.numpy()
    results.extend(outputs)


In [9]:
# Calculate accuracy
correct_predictions = 0
total_predictions = len(dataset_test)
for predicted, example in zip(results, dataset_test):
    predicted_label = predicted
    ground_truth_label = example['labels']
    if predicted_label == ground_truth_label:
        correct_predictions += 1

accuracy = correct_predictions / total_predictions

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 80.17%
